In [13]:
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

In [14]:
USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project), Mozilla/5.0 (Windows NT 10.0; Win64; x64), Chrome/91.0.4472.124 Safari/537.36")

In [15]:
def load_data(path='../data/ausflugziele'):
    data = []
    for filename in os.listdir(path):
        # Nur JSON-Dateien verarbeiten
        if filename.endswith('.json'):
            # Öffnen und Laden der JSON-Datei im Lesemodus
            with open(os.path.join(path, filename), 'r', encoding='utf-8') as f:
                # Inhalt der JSON-Datei laden
                content = json.load(f)
                if isinstance(content, list):
                    data.extend(content)
                else:
                    data.append(content)
    return data

data = load_data()
print(f'{len(data)} documents loaded.')

18 documents loaded.
18 documents loaded.


In [16]:
def clean_text(text):
     # Kleinbuchstaben, Entfernen von Sonderzeichen
    text = text.lower().replace(",", " ").replace("!", " ").replace("?", " ").replace("-", " ").replace(")", " ").strip()
    text = text.split()
    return text

In [28]:
docsListe = []
for i, doc in enumerate(data, start=1):
        if 'beschreibung' in doc:
            text = f"""
            Name: {doc.get('name')}
            Ort: {doc.get('ort')}
            Region: {doc.get('region')}
            Öffnungszeiten: {doc.get('oeffnungszeiten')}
            Eintrittspreise: {doc.get('eintrittspreise')}
            Zielgruppe: {doc.get('zielgruppe')}
            Kategorie: {doc.get('kategorie')}
            Beschreibung: {doc.get('beschreibung')}
            """
            doc_entry = {
                "name": doc.get("name"),
                "ort": doc.get("ort"),
                "region": doc.get("region"),
                "oeffnungszeiten": doc.get("oeffnungszeiten"),
                "eintrittspreise": doc.get("eintrittspreise"),
                "zielgruppe": doc.get("zielgruppe"),
                "kategorie": doc.get("kategorie"),
                "text_tokens": clean_text(doc['beschreibung']),  # tokenisierte Beschreibung
                "text": text.strip()
            }
            docsListe.append(doc_entry)

            # Vorschau ausgeben
            print(f"\n--- Dokument {i}: {doc.get('name', 'Unbekannt')} ---")
            print(f"\n{doc_entry['text'][:300]}...")  # nur die ersten 300 Zeichen

        else:
            print(f"Fehlende Beschreibung in Dokument: {doc.get('name', 'unknown')}")
print(f"{len(docsListe)} Dokumente für RAG vorbereitet.")


--- Dokument 1: Gedenkstätte und Museum Sachsenhausen ---

Name: Gedenkstätte und Museum Sachsenhausen
            Ort: Oranienburg
            Region: Brandenburg
            Öffnungszeiten: täglich 08:30 - 17:00 Uhr
            Eintrittspreise: Erwachsene: 10 €, Ermäßigt: 5 €, Kinder unter 18 Jahren: frei
            Zielgruppe: Erwachsene, Jugendliche, K...

--- Dokument 2: Museum Barberini ---

Name: Museum Barberini
            Ort: Potsdam
            Region: Brandenburg
            Öffnungszeiten: Dienstag bis Sonntag 10:00 - 19:00 Uhr, Dienstag geschlossen
            Eintrittspreise: Erwachsene: 16-18 Euro, Ermäßigt: 10 Euro, Kombiticket: 20 Euro, Kombiticket ermäßigt: 12 Euro,   Kind...

--- Dokument 3: Freilandmuseum Lehde ---

Name: Freilandmuseum Lehde
            Ort: Lübbenau, Luebbenau
            Region: Spreewald, Brandenburg
            Öffnungszeiten: täglich 10:00 - 18:00 Uhr
            Eintrittspreise: Erwachsene: 6 €, Kinder unter 18 Jahren: frei
            Zi

In [33]:
#embedding
#model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
texts = [doc['text'] for doc in docsListe]


embeddings_json = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings_json).astype('float32')
print(f"Embeddings für {len(embeddings)} Dokumente erstellt.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings für 18 Dokumente erstellt.


In [19]:
#Daten in Milvus einfügen
connections.connect("default", host="localhost", port="19530")


In [20]:
collection_name = "ausflug_collection"

if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="name", dtype=DataType.VARCHAR, max_length=512),
    FieldSchema(name="ort", dtype=DataType.VARCHAR, max_length=256),
    FieldSchema(name="region", dtype=DataType.VARCHAR, max_length=256)
]

schema = CollectionSchema(fields, description="Ausflugziele JSON Datenbank")

collection = Collection(collection_name, schema)

In [21]:
texts = [doc["text"] or "" for doc in docsListe]
names = [doc["name"] or "" for doc in docsListe]
orte = [doc["ort"] or "" for doc in docsListe]
regions = [doc["region"] or "" for doc in docsListe]

collection.insert([
    embeddings.tolist(),
    texts,
    names,
    orte,
    regions
])

(insert count: 18, delete count: 0, upsert count: 0, timestamp: 464847591036157956, success count: 18, err count: 0

(insert count: 18, delete count: 0, upsert count: 0, timestamp: 464847593080881156, success count: 18, err count: 0

In [22]:
#Index erstellen
index_params = {

    "index_type": "HNSW",
    "metric_type": "COSINE", #Ähnlichkeitsmaß für die Suche

    "params": {
        "M": 16, # Anzahl der Verbindungen pro Knoten
        "efConstruction": 200 # Genauigkeit vs. Geschwindigkeit beim Indexaufbau
    }
}

collection.create_index(
    field_name="embedding",
    index_params=index_params
)

collection.load()

In [23]:
def semantic_search_json(query, k=5):

    q_emb = model.encode([query]).astype("float32").tolist()

    results = collection.search(
        data=q_emb,
        anns_field="embedding",
        param={
            "metric_type": "COSINE",
            "params": {"ef": 50}
        },
        limit=k,
        output_fields=["text", "name", "ort", "region"]
    )

    output = []

    for hits in results:
        for hit in hits:
            output.append({
                "name": hit.entity.get("name"),
                "ort": hit.entity.get("ort"),
                "region": hit.entity.get("region"),
                "text": hit.entity.get("text"),
                "score": hit.score
            })

    return output

In [24]:
query = "Ausflug mit Kindern im Spreewald"

results = semantic_search_json(query, k=5)

for r in results:

    print(f"\n--- {r['name']} ---")
    print(f"Ort: {r['ort']} ({r['region']})")
    print(f"Score: {r['score']:.3f}")
    print(f"Text: {r['text'][:200]}...")


--- Spreewald-Museum Lübbenau ---
Ort: Lübbenau, Luebbenau (Spreewald, Brandenburg)
Score: 0.457
Text: Name: Spreewald-Museum Lübbenau
            Ort: Lübbenau, Luebbenau
            Region: Spreewald, Brandenburg
            Öffnungszeiten: täglich 11:00 - 16:00 Uhr
            Eintrittspreise: Erwac...

--- Heimatmuseum Dissen ---
Ort: Dissen-Striesow (Spreewald, Brandenburg)
Score: 0.378
Text: Name: Heimatmuseum Dissen
            Ort: Dissen-Striesow
            Region: Spreewald, Brandenburg
            Öffnungszeiten: Hauptsaison (ab Mai bis Ende Oktober): Dienstag - Donnerstag, 10:00 – ...

--- Großen Spreewaldhafen in Lübbenau ---
Ort:  ()
Score: 0.355
Text: Name: Großen Spreewaldhafen in Lübbenau
            Ort: None
            Region: None
            Öffnungszeiten: None
            Eintrittspreise: None
            Zielgruppe: None
            Kateg...

--- Freilandmuseum Lehde ---
Ort: Lübbenau, Luebbenau (Spreewald, Brandenburg)
Score: 0.342
Text: Name: Freilandmuseum